<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Selects the modeling population and columns from the diploma-merged table to produce the feature-engineering input dataset.

**Inputs / Data Sources:**
- `MERGE_DIR / "merged_with_diploma.parquet"` (written by `02_merge_diploma`)
- raw ADD / CRG tables via `RAW_DIR` (diagnostic cells only)

**Outputs / Side Effects:**
- `FEATURES_DIR / "selected_model_population.parquet"` (sole owner)

**Logic Flow:**
1. Load the diploma-merged table.
2. Inspect candidate columns and missingness.
3. Drop diagnostic/duplicate columns; filter rows missing `start_agpa_points`.
4. Assert `diploma_gpa`/`diploma_type_id` survive, then write the selected parquet.

**Maintainability Notes:** Column selection is denylist-based (logged tech debt: convert to allowlist); schema changes upstream can silently drop or reshape model inputs.

In [1]:
import pandas as pd
from src.paths import MERGE_DIR, RAW_DIR, FEATURES_DIR, assert_data_root
from src.io_utils import save_parquet

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [3]:
MERGED_WITH_DIPLOMA_PATH = MERGE_DIR / "merged_with_diploma.parquet"
assert_data_root(MERGED_WITH_DIPLOMA_PATH)
df = pd.read_parquet(MERGED_WITH_DIPLOMA_PATH)

In [4]:
df.head()

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,points,finish_status,course_outcome_status,register_status,in_credits,in_gpa,in_agpa,study_mode,course_credits,course_name_sl,degree_name_sl,attempt_number,attempt_count,student_status_id,prev_gpa_points,prev_gpa_percent,gpa_points,start_agpa_points,start_agpa_percent,total_semesters,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_id,start_level_name_pl,start_part_id,finish_part_id,finish_status_add_snapshot,has_add_snapshot,crg_degree_id,acd_degree_ids,acd_degree_count,issue_type,resolved_acd_degree_id,recommended_action,student_degree_id_original,degree_id_for_acd_merge,degree_id_corrected_for_acd,acd_resolution_rule,degree_course_id,acd_degree_id,requirement_type_id,requirement_type_sl,acd_course_credits,degree_requirement_credits_count,acd_course_name_sl,acd_degree_name_sl,has_degree_course_info,acd_match_type,diploma_gpa,diploma_type_id
0,939272.111,10000.111,1016.111,20152,3.111,5.111,677.111,82,3.0,P,passed,R,Y,Y,Y,C,2.0,المنطق والتفكير العلمي,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,206.111,Second Year,20151,20191,GRADUATED,True,3.111,"[13.111, 1.111, 2.111, 21.111, 5.111, 4.111, 6.111, 3.111, 11.111, 9.111, 12.111, 8.111, 7.111, 10.111, 22.111, 24.1...",29,valid_degree_course_pair,3.111,keep_original_degree_id,3.111,3.111,False,original_degree_id,764.111,3.111,2,متطلبات الجامعة الاختيارية,2.0,6,المنطق والتفكير العلمي,هندسة البرمجيات ونظم المعلومات,True,exact_degree_course_match,74.71,15
1,953865.111,10000.111,1017.111,20153,3.111,5.111,680.111,67,2.25,P,passed,R,Y,Y,Y,C,2.0,لغة اجنبية حية (ثانية),هندسة البرمجيات ونظم المعلومات,1,1,155349.111,2.36,67.2,2.25,2.39,67.8,3.0,12.0,35.0,9.0,3.0,9.0,35.0,0.0,3.0,206.111,Second Year,20151,20191,GRADUATED,True,3.111,"[13.111, 1.111, 2.111, 21.111, 5.111, 4.111, 6.111, 3.111, 11.111, 9.111, 12.111, 8.111, 7.111, 10.111, 22.111, 24.1...",29,valid_degree_course_pair,3.111,keep_original_degree_id,3.111,3.111,False,original_degree_id,765.111,3.111,2,متطلبات الجامعة الاختيارية,2.0,6,لغة اجنبية حية (ثانية),هندسة البرمجيات ونظم المعلومات,True,exact_degree_course_match,74.71,15
2,1076397.111,10000.111,1019.111,20171,3.111,5.111,981.111,66,2.25,P,passed,R,Y,Y,Y,C,2.0,مدخل الى القانون,هندسة البرمجيات ونظم المعلومات,1,1,240678.111,2.17,63.4,1.76,2.28,65.6,6.0,27.0,80.0,17.0,6.0,17.0,80.0,0.0,6.0,348.111,Fourth Year,20151,20191,GRADUATED,True,3.111,"[13.111, 1.111, 2.111, 21.111, 5.111, 4.111, 6.111, 3.111, 11.111, 9.111, 12.111, 8.111, 7.111, 10.111, 22.111, 24.1...",29,valid_degree_course_pair,3.111,keep_original_degree_id,3.111,3.111,False,original_degree_id,768.111,3.111,2,متطلبات الجامعة الاختيارية,2.0,6,مدخل الى القانون,هندسة البرمجيات ونظم المعلومات,True,exact_degree_course_match,74.71,15
3,925010.111,10000.111,431.111,20152,3.111,5.111,677.111,83,3.0,P,passed,R,Y,Y,Y,C,3.0,الرياضيات المتقطعة,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,206.111,Second Year,20151,20191,GRADUATED,True,3.111,"[5.111, 4.111, 21.111, 6.111, 3.111]",5,valid_degree_course_pair,3.111,keep_original_degree_id,3.111,3.111,False,original_degree_id,773.111,3.111,3,متطلبات الكلية الإجبارية,3.0,65,الرياضيات المتقطعة,هندسة البرمجيات ونظم المعلومات,True,exact_degree_course_match,74.71,15
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,2.0,P,passed,R,Y,Y,Y,C,4.0,الجبر الخطي ونظرية المصفوفات,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,3.111,"[5.111, 4.111, 21.111, 6.111, 3.111]",5,valid_degree_course_pair,3.111,keep_original_degree_id,3.111,3.111,False,original_degree_id,774.111,3.111,3,متطلبات الكلية الإجبارية,4.0,65,الجبر الخطي ونظرية المصفوفات,هند

In [5]:
df[df['student_id']==10428.111]

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,points,finish_status,course_outcome_status,register_status,in_credits,in_gpa,in_agpa,study_mode,course_credits,course_name_sl,degree_name_sl,attempt_number,attempt_count,student_status_id,prev_gpa_points,prev_gpa_percent,gpa_points,start_agpa_points,start_agpa_percent,total_semesters,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_id,start_level_name_pl,start_part_id,finish_part_id,finish_status_add_snapshot,has_add_snapshot,crg_degree_id,acd_degree_ids,acd_degree_count,issue_type,resolved_acd_degree_id,recommended_action,student_degree_id_original,degree_id_for_acd_merge,degree_id_corrected_for_acd,acd_resolution_rule,degree_course_id,acd_degree_id,requirement_type_id,requirement_type_sl,acd_course_credits,degree_requirement_credits_count,acd_course_name_sl,acd_degree_name_sl,has_degree_course_info,acd_match_type,diploma_gpa,diploma_type_id


In [6]:
df.columns

Index(['student_course_id', 'student_id', 'course_id', 'part_id', 'degree_id', 'faculty_id', 'grade_id', 'final_mark', 'points', 'finish_status',
       'course_outcome_status', 'register_status', 'in_credits', 'in_gpa', 'in_agpa', 'study_mode', 'course_credits', 'course_name_sl', 'degree_name_sl',
       'attempt_number', 'attempt_count', 'student_status_id', 'prev_gpa_points', 'prev_gpa_percent', 'gpa_points', 'start_agpa_points', 'start_agpa_percent',
       'total_semesters', 'start_total_in_courses', 'start_total_in_credits', 'semester_reg_credits', 'semester_reg_courses', 'semester_pass_credits',
       'total_pass_credits', 'total_fail_credits', 'reg_total_semesters', 'start_level_id', 'start_level_name_pl', 'start_part_id', 'finish_part_id',
       'finish_status_add_snapshot', 'has_add_snapshot', 'crg_degree_id', 'acd_degree_ids', 'acd_degree_count', 'issue_type', 'resolved_acd_degree_id',
       'recommended_action', 'student_degree_id_original', 'degree_id_for_acd_merge', 'd

In [7]:
# df_dropped=df.drop(columns=['student_course_id', 'student_id', 'course_id','degree_id',
#        'faculty_id','student_status_id','crg_degree_id',])

In [8]:
diagnostic_cols = [
    "crg_degree_id", "acd_degree_ids", "acd_degree_count",
    "issue_type", "resolved_acd_degree_id", "recommended_action",
    "student_degree_id_original", "degree_id_for_acd_merge",
    "degree_id_corrected_for_acd", "acd_resolution_rule",
    "acd_degree_id", "degree_course_id",
    "acd_course_name_sl", "acd_degree_name_sl",  # duplicates of course_name_sl, degree_name_sl
]
df1 = df.drop(columns=[c for c in diagnostic_cols if c in df])
# 55 cols → ~41 cols

In [9]:
df1.columns

Index(['student_course_id', 'student_id', 'course_id', 'part_id', 'degree_id', 'faculty_id', 'grade_id', 'final_mark', 'points', 'finish_status',
       'course_outcome_status', 'register_status', 'in_credits', 'in_gpa', 'in_agpa', 'study_mode', 'course_credits', 'course_name_sl', 'degree_name_sl',
       'attempt_number', 'attempt_count', 'student_status_id', 'prev_gpa_points', 'prev_gpa_percent', 'gpa_points', 'start_agpa_points', 'start_agpa_percent',
       'total_semesters', 'start_total_in_courses', 'start_total_in_credits', 'semester_reg_credits', 'semester_reg_courses', 'semester_pass_credits',
       'total_pass_credits', 'total_fail_credits', 'reg_total_semesters', 'start_level_id', 'start_level_name_pl', 'start_part_id', 'finish_part_id',
       'finish_status_add_snapshot', 'has_add_snapshot', 'requirement_type_id', 'requirement_type_sl', 'acd_course_credits',
       'degree_requirement_credits_count', 'has_degree_course_info', 'acd_match_type', 'diploma_gpa', 'diploma_type

In [10]:
df1 = df1.dropna(subset=['start_agpa_points'])

In [11]:
df1.info()

<class 'pandas.DataFrame'>
Index: 784245 entries, 0 to 784298
Data columns (total 50 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   student_course_id                 784245 non-null  string  
 1   student_id                        784245 non-null  string  
 2   course_id                         784245 non-null  str     
 3   part_id                           784245 non-null  Int64   
 4   degree_id                         784245 non-null  str     
 5   faculty_id                        784245 non-null  string  
 6   grade_id                          784245 non-null  string  
 7   final_mark                        784245 non-null  Int64   
 8   points                            784245 non-null  Float64 
 9   finish_status                     784245 non-null  category
 10  course_outcome_status             784245 non-null  category
 11  register_status                   784245 non-null  cate

In [12]:
FEATURE_COLS_BASELINE = [
    "prev_gpa_points",
    "prev_gpa_percent",
    "start_agpa_points",
    "start_agpa_percent",
    "registered_academic_semesters",
    "inactive_semester_gap",
    "has_inactive_semester_gap",
    "start_level_id",
    "degree_id",
    "course_id",
    "semester_num",
    "requirement_type_id_model",
    "course_credits_model",
    "degree_requirement_credits_count_model",
    "has_degree_course_info",
    "has_trusted_acd_features",
    "acd_match_type",
    "attempt_number",
    "in_credits",
    "in_gpa",
    "in_agpa",
]

In [13]:
df1.head()

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,points,finish_status,course_outcome_status,register_status,in_credits,in_gpa,in_agpa,study_mode,course_credits,course_name_sl,degree_name_sl,attempt_number,attempt_count,student_status_id,prev_gpa_points,prev_gpa_percent,gpa_points,start_agpa_points,start_agpa_percent,total_semesters,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_id,start_level_name_pl,start_part_id,finish_part_id,finish_status_add_snapshot,has_add_snapshot,requirement_type_id,requirement_type_sl,acd_course_credits,degree_requirement_credits_count,has_degree_course_info,acd_match_type,diploma_gpa,diploma_type_id
0,939272.111,10000.111,1016.111,20152,3.111,5.111,677.111,82,3.0,P,passed,R,Y,Y,Y,C,2.0,المنطق والتفكير العلمي,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,206.111,Second Year,20151,20191,GRADUATED,True,2,متطلبات الجامعة الاختيارية,2.0,6,True,exact_degree_course_match,74.71,15
1,953865.111,10000.111,1017.111,20153,3.111,5.111,680.111,67,2.25,P,passed,R,Y,Y,Y,C,2.0,لغة اجنبية حية (ثانية),هندسة البرمجيات ونظم المعلومات,1,1,155349.111,2.36,67.2,2.25,2.39,67.8,3.0,12.0,35.0,9.0,3.0,9.0,35.0,0.0,3.0,206.111,Second Year,20151,20191,GRADUATED,True,2,متطلبات الجامعة الاختيارية,2.0,6,True,exact_degree_course_match,74.71,15
2,1076397.111,10000.111,1019.111,20171,3.111,5.111,981.111,66,2.25,P,passed,R,Y,Y,Y,C,2.0,مدخل الى القانون,هندسة البرمجيات ونظم المعلومات,1,1,240678.111,2.17,63.4,1.76,2.28,65.6,6.0,27.0,80.0,17.0,6.0,17.0,80.0,0.0,6.0,348.111,Fourth Year,20151,20191,GRADUATED,True,2,متطلبات الجامعة الاختيارية,2.0,6,True,exact_degree_course_match,74.71,15
3,925010.111,10000.111,431.111,20152,3.111,5.111,677.111,83,3.0,P,passed,R,Y,Y,Y,C,3.0,الرياضيات المتقطعة,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,206.111,Second Year,20151,20191,GRADUATED,True,3,متطلبات الكلية الإجبارية,3.0,65,True,exact_degree_course_match,74.71,15
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,2.0,P,passed,R,Y,Y,Y,C,4.0,الجبر الخطي ونظرية المصفوفات,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,3,متطلبات الكلية الإجبارية,4.0,65,True,exact_degree_course_match,74.71,15


In [14]:
df1[(df1['student_id'].eq('10000.111')&df1['part_id'].eq(20151))]

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,points,finish_status,course_outcome_status,register_status,in_credits,in_gpa,in_agpa,study_mode,course_credits,course_name_sl,degree_name_sl,attempt_number,attempt_count,student_status_id,prev_gpa_points,prev_gpa_percent,gpa_points,start_agpa_points,start_agpa_percent,total_semesters,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_id,start_level_name_pl,start_part_id,finish_part_id,finish_status_add_snapshot,has_add_snapshot,requirement_type_id,requirement_type_sl,acd_course_credits,degree_requirement_credits_count,has_degree_course_info,acd_match_type,diploma_gpa,diploma_type_id
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,2.0,P,passed,R,Y,Y,Y,C,4.0,الجبر الخطي ونظرية المصفوفات,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,3,متطلبات الكلية الإجبارية,4.0,65,True,exact_degree_course_match,74.71,15
16,901338.111,10000.111,501.111,20151,3.111,5.111,680.111,67,2.25,P,passed,R,Y,Y,Y,C,3.0,الفيزياء 1,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,3,متطلبات الكلية الإجبارية,3.0,65,True,exact_degree_course_match,74.71,15
26,901342.111,10000.111,513.111,20151,3.111,5.111,677.111,84,3.0,P,passed,R,Y,Y,Y,C,3.0,مقدمة في الخوارزميات والبرمجة,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,3,متطلبات الكلية الإجبارية,3.0,65,True,exact_degree_course_match,74.71,15
56,901337.111,10000.111,955.111,20151,3.111,5.111,681.111,63,2.0,P,passed,R,Y,Y,Y,C,2.0,اللغة العربية,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,1,متطلبات الجامعة الإجبارية,2.0,9,True,exact_degree_course_match,74.71,15
57,901341.111,10000.111,956.111,20151,3.111,5.111,679.111,71,2.5,P,passed,R,Y,Y,Y,C,2.0,اللغة الانكليزية 1,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,1,متطلبات الجامعة الإجبارية,2.0,9,True,exact_degree_course_match,74.71,15
59,901339.111,10000.111,967.111,20151,3.111,5.111,678.111,78,2.75,P,passed,R,Y,Y,Y,C,3.0,مهارات الحاسوب,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,1,متطلبات الجامعة الإجبارية,3.0,9,True,exact_degree_course_match,74.71,15


In [15]:
# drop has_add_snapshot start_level_id study mode acd_course_credits acd_match_type student_course_id student_id faculty_id grade_id

In [16]:
df_feature=df1.drop(columns=['has_add_snapshot','start_level_id','study_mode','acd_course_credits','acd_match_type'])
# ','student_course_id','student_id','faculty_id','grade_id','student_status_id'

In [17]:
df1['has_add_snapshot'].value_counts()

has_add_snapshot
True    784245
Name: count, dtype: int64

In [18]:
df1['start_level_id'].value_counts()

start_level_id
569.111    23491
617.111    16038
597.111    15452
169.111    14873
596.111    14764
           ...  
834.111        5
2.111          5
337.111        2
818.111        2
692.111        1
Name: count, Length: 455, dtype: int64[pyarrow]

In [19]:
df1['has_degree_course_info'].value_counts()

has_degree_course_info
True     782525
False      1720
Name: count, dtype: int64

In [20]:
import numpy as np

In [21]:
df1['in_credits'].value_counts()

in_credits
Y    779050
N      5195
Name: count, dtype: int64

In [22]:
df1['in_gpa'].value_counts()

in_gpa
Y    779118
N      5127
Name: count, dtype: int64

In [23]:
df1.head()

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,points,finish_status,course_outcome_status,register_status,in_credits,in_gpa,in_agpa,study_mode,course_credits,course_name_sl,degree_name_sl,attempt_number,attempt_count,student_status_id,prev_gpa_points,prev_gpa_percent,gpa_points,start_agpa_points,start_agpa_percent,total_semesters,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_id,start_level_name_pl,start_part_id,finish_part_id,finish_status_add_snapshot,has_add_snapshot,requirement_type_id,requirement_type_sl,acd_course_credits,degree_requirement_credits_count,has_degree_course_info,acd_match_type,diploma_gpa,diploma_type_id
0,939272.111,10000.111,1016.111,20152,3.111,5.111,677.111,82,3.0,P,passed,R,Y,Y,Y,C,2.0,المنطق والتفكير العلمي,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,206.111,Second Year,20151,20191,GRADUATED,True,2,متطلبات الجامعة الاختيارية,2.0,6,True,exact_degree_course_match,74.71,15
1,953865.111,10000.111,1017.111,20153,3.111,5.111,680.111,67,2.25,P,passed,R,Y,Y,Y,C,2.0,لغة اجنبية حية (ثانية),هندسة البرمجيات ونظم المعلومات,1,1,155349.111,2.36,67.2,2.25,2.39,67.8,3.0,12.0,35.0,9.0,3.0,9.0,35.0,0.0,3.0,206.111,Second Year,20151,20191,GRADUATED,True,2,متطلبات الجامعة الاختيارية,2.0,6,True,exact_degree_course_match,74.71,15
2,1076397.111,10000.111,1019.111,20171,3.111,5.111,981.111,66,2.25,P,passed,R,Y,Y,Y,C,2.0,مدخل الى القانون,هندسة البرمجيات ونظم المعلومات,1,1,240678.111,2.17,63.4,1.76,2.28,65.6,6.0,27.0,80.0,17.0,6.0,17.0,80.0,0.0,6.0,348.111,Fourth Year,20151,20191,GRADUATED,True,2,متطلبات الجامعة الاختيارية,2.0,6,True,exact_degree_course_match,74.71,15
3,925010.111,10000.111,431.111,20152,3.111,5.111,677.111,83,3.0,P,passed,R,Y,Y,Y,C,3.0,الرياضيات المتقطعة,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,206.111,Second Year,20151,20191,GRADUATED,True,3,متطلبات الكلية الإجبارية,3.0,65,True,exact_degree_course_match,74.71,15
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,2.0,P,passed,R,Y,Y,Y,C,4.0,الجبر الخطي ونظرية المصفوفات,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,205.111,First Year,20151,20191,GRADUATED,True,3,متطلبات الكلية الإجبارية,4.0,65,True,exact_degree_course_match,74.71,15


In [24]:
df_feature.head()

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,points,finish_status,course_outcome_status,register_status,in_credits,in_gpa,in_agpa,course_credits,course_name_sl,degree_name_sl,attempt_number,attempt_count,student_status_id,prev_gpa_points,prev_gpa_percent,gpa_points,start_agpa_points,start_agpa_percent,total_semesters,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_name_pl,start_part_id,finish_part_id,finish_status_add_snapshot,requirement_type_id,requirement_type_sl,degree_requirement_credits_count,has_degree_course_info,diploma_gpa,diploma_type_id
0,939272.111,10000.111,1016.111,20152,3.111,5.111,677.111,82,3.0,P,passed,R,Y,Y,Y,2.0,المنطق والتفكير العلمي,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,20191,GRADUATED,2,متطلبات الجامعة الاختيارية,6,True,74.71,15
1,953865.111,10000.111,1017.111,20153,3.111,5.111,680.111,67,2.25,P,passed,R,Y,Y,Y,2.0,لغة اجنبية حية (ثانية),هندسة البرمجيات ونظم المعلومات,1,1,155349.111,2.36,67.2,2.25,2.39,67.8,3.0,12.0,35.0,9.0,3.0,9.0,35.0,0.0,3.0,Second Year,20151,20191,GRADUATED,2,متطلبات الجامعة الاختيارية,6,True,74.71,15
2,1076397.111,10000.111,1019.111,20171,3.111,5.111,981.111,66,2.25,P,passed,R,Y,Y,Y,2.0,مدخل الى القانون,هندسة البرمجيات ونظم المعلومات,1,1,240678.111,2.17,63.4,1.76,2.28,65.6,6.0,27.0,80.0,17.0,6.0,17.0,80.0,0.0,6.0,Fourth Year,20151,20191,GRADUATED,2,متطلبات الجامعة الاختيارية,6,True,74.71,15
3,925010.111,10000.111,431.111,20152,3.111,5.111,677.111,83,3.0,P,passed,R,Y,Y,Y,3.0,الرياضيات المتقطعة,هندسة البرمجيات ونظم المعلومات,1,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,20191,GRADUATED,3,متطلبات الكلية الإجبارية,65,True,74.71,15
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,2.0,P,passed,R,Y,Y,Y,4.0,الجبر الخطي ونظرية المصفوفات,هندسة البرمجيات ونظم المعلومات,1,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,First Year,20151,20191,GRADUATED,3,متطلبات الكلية الإجبارية,65,True,74.71,15


In [25]:
#register_status  in_credits	in_gpa	in_agpa	course_name_sl	degree_name_sl	 finish_status  attempt_count requirement_type_sl has_degree_course_info finish_part_id course_outcome_status

In [26]:
#points  finish_status finish_status_add_snapshot finish_part_id

In [27]:
df_feature2=df_feature.drop(columns=['register_status','in_credits','in_gpa','in_agpa','course_name_sl','degree_name_sl','finish_status','attempt_count','requirement_type_sl','has_degree_course_info','finish_part_id','finish_status_add_snapshot','points','course_outcome_status'])

In [28]:
df_feature.info()

<class 'pandas.DataFrame'>
Index: 784245 entries, 0 to 784298
Data columns (total 45 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   student_course_id                 784245 non-null  string  
 1   student_id                        784245 non-null  string  
 2   course_id                         784245 non-null  str     
 3   part_id                           784245 non-null  Int64   
 4   degree_id                         784245 non-null  str     
 5   faculty_id                        784245 non-null  string  
 6   grade_id                          784245 non-null  string  
 7   final_mark                        784245 non-null  Int64   
 8   points                            784245 non-null  Float64 
 9   finish_status                     784245 non-null  category
 10  course_outcome_status             784245 non-null  category
 11  register_status                   784245 non-null  cate

In [29]:
df_feature['register_status'].value_counts()

register_status
R    769601
E     14644
Name: count, dtype: int64

In [30]:
df_feature2.head()

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,course_credits,attempt_number,student_status_id,prev_gpa_points,prev_gpa_percent,gpa_points,start_agpa_points,start_agpa_percent,total_semesters,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_name_pl,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
0,939272.111,10000.111,1016.111,20152,3.111,5.111,677.111,82,2.0,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,2,6,74.71,15
1,953865.111,10000.111,1017.111,20153,3.111,5.111,680.111,67,2.0,1,155349.111,2.36,67.2,2.25,2.39,67.8,3.0,12.0,35.0,9.0,3.0,9.0,35.0,0.0,3.0,Second Year,20151,2,6,74.71,15
2,1076397.111,10000.111,1019.111,20171,3.111,5.111,981.111,66,2.0,1,240678.111,2.17,63.4,1.76,2.28,65.6,6.0,27.0,80.0,17.0,6.0,17.0,80.0,0.0,6.0,Fourth Year,20151,2,6,74.71,15
3,925010.111,10000.111,431.111,20152,3.111,5.111,677.111,83,3.0,1,155348.111,2.41,68.2,2.36,2.41,68.2,2.0,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,3,65,74.71,15
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,4.0,1,155347.111,<NA>,<NA>,2.41,0.0,0.0,1.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,First Year,20151,3,65,74.71,15


In [31]:
##prev_gpa_percent start_agpa_percent total_semesters

In [32]:
df_feature2 = df_feature2.drop(columns=['prev_gpa_percent', 'start_agpa_percent', 'total_semesters'])

In [33]:
df_feature2.head()

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,course_credits,attempt_number,student_status_id,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_name_pl,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
0,939272.111,10000.111,1016.111,20152,3.111,5.111,677.111,82,2.0,1,155348.111,2.41,2.36,2.41,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,2,6,74.71,15
1,953865.111,10000.111,1017.111,20153,3.111,5.111,680.111,67,2.0,1,155349.111,2.36,2.25,2.39,12.0,35.0,9.0,3.0,9.0,35.0,0.0,3.0,Second Year,20151,2,6,74.71,15
2,1076397.111,10000.111,1019.111,20171,3.111,5.111,981.111,66,2.0,1,240678.111,2.17,1.76,2.28,27.0,80.0,17.0,6.0,17.0,80.0,0.0,6.0,Fourth Year,20151,2,6,74.71,15
3,925010.111,10000.111,431.111,20152,3.111,5.111,677.111,83,3.0,1,155348.111,2.41,2.36,2.41,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,3,65,74.71,15
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,4.0,1,155347.111,<NA>,2.41,0.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,First Year,20151,3,65,74.71,15


In [34]:
df_feature2.describe()

,part_id,final_mark,course_credits,attempt_number,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
count,784245.0,784245.0,784245.0,784245.0,692124.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,781062.0,784245.0,782525.0,782525.0,784245.000000,784189.0
mean,20199.892257,67.676301,2.858021,1.204977,2.108644,2.152532,1.883961,25.606995,70.529081,15.972688,5.805837,13.461523,72.564906,15.82112,7.537082,20178.52777,3.635557,114.302079,84.425412,15.530551
std,37.801603,17.099574,1.294855,0.629768,0.883413,0.806815,1.009606,19.930176,55.246798,4.521095,1.704155,5.585118,57.137499,30.916,5.502418,38.352582,1.274985,71.613799,11.460545,4.277855
min,20051.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20051.0,1.0,4.0,8.570000,9.0
25%,20172.0,58.0,2.0,1.0,1.63,1.68,1.58,8.0,21.0,14.0,5.0,9.0,21.0,0.0,3.0,20151.0,3.0,48.0,76.250000,15.0
50%,20203.0,69.0,3.0,1.0,2.25,2.28,2.15,22.0,62.0,17.0,6.0,15.0,64.0,4.0,7.0,20181.0,3.0,129.0,86.630000,15.0
75%,20231.0,80.0,3.0,1.0,2.75,2.74,2.55,41.0,112.0,18.0,7.0,18.0,115.0,18.0,11.0,20211.0,5.0,170.0,94.130000,15.0
max,20253.0,100.0,24.0,16.0,4.0,4.0,4.0,81.0,252.0,72.5,17.0,49.0,334.0,713.5,50.0,20252.0,6.0,236.0,100.000000,71.0


In [35]:
df_feature2[df_feature2['total_fail_credits'].eq(713.5)]

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,course_credits,attempt_number,student_status_id,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_name_pl,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
617418,2573279.111,3470.111,1093.111,20232,2.111,2.111,978.111,81,24.0,1,371794.111,0.0,3.0,2.11,62.0,227.0,24.0,1.0,24.0,260.5,713.5,49.0,Sixth Year,20161,5,210,76.67,15
617422,2201832.111,3470.111,664.111,20213,2.111,2.111,974.111,99,3.0,1,321187.111,0.0,4.0,2.06,61.0,224.0,3.0,1.0,3.0,257.5,713.5,46.0,Sixth Year,20161,3,22,76.67,15
617435,1549096.111,3470.111,689.111,20203,2.111,2.111,982.111,60,4.0,5,301185.111,0.67,1.88,1.57,56.0,198.0,8.0,2.0,8.0,231.5,713.5,43.0,Sixth Year,20161,5,210,76.67,15
617439,1549095.111,3470.111,690.111,20203,2.111,2.111,983.111,59,4.0,4,301185.111,0.67,1.88,1.57,56.0,198.0,8.0,2.0,8.0,231.5,713.5,43.0,Sixth Year,20161,5,210,76.67,15
617446,1559322.111,3470.111,696.111,20211,2.111,2.111,981.111,67,6.0,5,308846.111,1.88,2.25,1.7,58.0,206.0,18.0,3.0,18.0,239.5,713.5,44.0,Sixth Year,20161,5,210,76.67,15
617459,1559323.111,3470.111,699.111,20211,2.111,2.111,981.111,66,6.0,7,308846.111,1.88,2.25,1.7,58.0,206.0,18.0,3.0,18.0,239.5,713.5,44.0,Sixth Year,20161,5,210,76.67,15
617474,1559324.111,3470.111,709.111,20211,2.111,2.111,981.111,67,6.0,5,308846.111,1.88,2.25,1.7,58.0,206.0,18.0,3.0,18.0,239.5,713.5,44.0,Sixth Year,20161,5,210,76.67,15


In [36]:
df_feature2.info()

<class 'pandas.DataFrame'>
Index: 784245 entries, 0 to 784298
Data columns (total 28 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   student_course_id                 784245 non-null  string 
 1   student_id                        784245 non-null  string 
 2   course_id                         784245 non-null  str    
 3   part_id                           784245 non-null  Int64  
 4   degree_id                         784245 non-null  str    
 5   faculty_id                        784245 non-null  string 
 6   grade_id                          784245 non-null  string 
 7   final_mark                        784245 non-null  Int64  
 8   course_credits                    784245 non-null  Float64
 9   attempt_number                    784245 non-null  Int64  
 10  student_status_id                 784245 non-null  string 
 11  prev_gpa_points                   692124 non-null  Float64
 12  gpa_

In [37]:
df_feature2.isna().sum()

student_course_id                       0
student_id                              0
course_id                               0
part_id                                 0
degree_id                               0
faculty_id                              0
grade_id                                0
final_mark                              0
course_credits                          0
attempt_number                          0
student_status_id                       0
prev_gpa_points                     92121
gpa_points                              0
start_agpa_points                       0
start_total_in_courses                  0
start_total_in_credits                  0
semester_reg_credits                    0
semester_reg_courses                    0
semester_pass_credits                   0
total_pass_credits                      0
total_fail_credits                      0
reg_total_semesters                  3183
start_level_name_pl                     0
start_part_id                     

In [38]:
df_feature2[df_feature2['course_credits'].eq(24)]

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,course_credits,attempt_number,student_status_id,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_name_pl,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
207,2573242.111,10003.111,1093.111,20232,2.111,2.111,983.111,59,24.0,1,371766.111,3.0,2.0,2.17,62.0,227.0,24.0,1.0,24.0,234.0,167.5,28.0,Sixth Year,20151,5,210,86.46,15
870,2267154.111,10015.111,1093.111,20221,2.111,2.111,977.111,89,24.0,1,329042.111,2.25,3.25,2.14,61.0,224.0,27.0,2.0,27.0,227.0,76.5,23.0,Sixth Year,20151,5,210,83.50,15
949,2182266.111,10017.111,1093.111,20212,2.111,2.111,980.111,73,24.0,1,320044.111,0.0,2.67,2.27,61.0,224.0,27.0,2.0,27.0,251.5,22.5,19.0,Sixth Year,20151,5,210,96.37,15
1022,2267157.111,10018.111,1093.111,20221,2.111,2.111,982.111,64,24.0,1,333763.111,0.0,2.22,2.2,61.0,224.0,27.0,2.0,27.0,224.0,85.0,23.0,Sixth Year,20151,5,210,90.14,15
1106,2267156.111,10019.111,1093.111,20221,2.111,2.111,980.111,74,24.0,1,333817.111,3.0,2.5,2.17,63.0,229.0,24.0,1.0,24.0,243.0,38.0,22.0,Sixth Year,20151,5,210,84.50,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
781732,2182264.111,9967.111,1093.111,20212,2.111,2.111,979.111,75,24.0,1,320361.111,0.0,2.69,2.56,61.0,224.0,27.0,2.0,27.0,224.0,20.0,17.0,Sixth Year,20151,5,210,88.71,15
782804,1843235.111,9978.111,1093.111,20211,2.111,2.111,978.111,84,24.0,1,309268.111,3.25,3.01,2.88,59.0,219.0,32.0,4.0,32.0,219.0,0.0,14.0,Sixth Year,20151,5,210,88.33,15
783706,1843285.111,9991.111,1093.111,20211,2.111,2.111,977.111,88,24.0,1,309198.111,3.0,3.25,2.68,61.0,224.0,27.0,2.0,27.0,209.0,14.0,16.0,Sixth Year,20151,5,210,92.46,15
783768,1843225.111,9992.111,1093.111,20211,2.111,2.111,979.111,78,24.0,1,310112.111,0.0,2.81,2.48,61.0,224.0,27.0,2.0,27.0,224.0,0.0,13.0,Sixth Year,20151,5,210,93.58,15


In [39]:
# degree_id course_id ## combine

In [40]:
# part_id # split start_part_id

In [41]:
# requirement_type_id	degree_requirement_credits_count?

In [42]:
## ordinry incoding 

In [43]:
# final mark drop 

In [44]:
a=df_feature2['semester_reg_credits'].value_counts()

In [45]:
a[1].max()

np.int64(170)

In [46]:
df_feature2[df_feature2['semester_reg_credits'].eq(72.5)]

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,course_credits,attempt_number,student_status_id,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_name_pl,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
617499,1049771.111,3472.111,676.111,20164,2.111,2.111,985.111,31,5.0,3,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617515,1048689.111,3472.111,687.111,20164,2.111,2.111,985.111,47,4.0,3,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617521,1045274.111,3472.111,689.111,20164,2.111,2.111,985.111,29,4.0,2,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617530,1045278.111,3472.111,690.111,20164,2.111,2.111,985.111,17,4.0,4,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617542,1048690.111,3472.111,691.111,20164,2.111,2.111,985.111,38,2.0,4,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617548,1045280.111,3472.111,692.111,20164,2.111,2.111,985.111,41,3.0,2,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617563,1048686.111,3472.111,696.111,20164,2.111,2.111,985.111,11,6.0,3,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617572,1045281.111,3472.111,697.111,20164,2.111,2.111,985.111,39,6.0,4,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617579,1048687.111,3472.111,699.111,20164,2.111,2.111,985.111,29,6.0,3,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15
617586,1048684.111,3472.111,702.111,20164,2.111,2.111,985.111,40,3.0,3,236604.111,0.0,0.12,0.95,42.0,141.5,72.5,16.0,6.0,152.5,333.0,26.0,Fifth year,20131,5,210,92.5,15


In [47]:
df_feature2.describe()

,part_id,final_mark,course_credits,attempt_number,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
count,784245.0,784245.0,784245.0,784245.0,692124.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,784245.0,781062.0,784245.0,782525.0,782525.0,784245.000000,784189.0
mean,20199.892257,67.676301,2.858021,1.204977,2.108644,2.152532,1.883961,25.606995,70.529081,15.972688,5.805837,13.461523,72.564906,15.82112,7.537082,20178.52777,3.635557,114.302079,84.425412,15.530551
std,37.801603,17.099574,1.294855,0.629768,0.883413,0.806815,1.009606,19.930176,55.246798,4.521095,1.704155,5.585118,57.137499,30.916,5.502418,38.352582,1.274985,71.613799,11.460545,4.277855
min,20051.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20051.0,1.0,4.0,8.570000,9.0
25%,20172.0,58.0,2.0,1.0,1.63,1.68,1.58,8.0,21.0,14.0,5.0,9.0,21.0,0.0,3.0,20151.0,3.0,48.0,76.250000,15.0
50%,20203.0,69.0,3.0,1.0,2.25,2.28,2.15,22.0,62.0,17.0,6.0,15.0,64.0,4.0,7.0,20181.0,3.0,129.0,86.630000,15.0
75%,20231.0,80.0,3.0,1.0,2.75,2.74,2.55,41.0,112.0,18.0,7.0,18.0,115.0,18.0,11.0,20211.0,5.0,170.0,94.130000,15.0
max,20253.0,100.0,24.0,16.0,4.0,4.0,4.0,81.0,252.0,72.5,17.0,49.0,334.0,713.5,50.0,20252.0,6.0,236.0,100.000000,71.0


In [48]:
df_feature2.info()

<class 'pandas.DataFrame'>
Index: 784245 entries, 0 to 784298
Data columns (total 28 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   student_course_id                 784245 non-null  string 
 1   student_id                        784245 non-null  string 
 2   course_id                         784245 non-null  str    
 3   part_id                           784245 non-null  Int64  
 4   degree_id                         784245 non-null  str    
 5   faculty_id                        784245 non-null  string 
 6   grade_id                          784245 non-null  string 
 7   final_mark                        784245 non-null  Int64  
 8   course_credits                    784245 non-null  Float64
 9   attempt_number                    784245 non-null  Int64  
 10  student_status_id                 784245 non-null  string 
 11  prev_gpa_points                   692124 non-null  Float64
 12  gpa_

In [49]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

# # تفعيل الطابع البصري الأنيق للمخططات
# sns.set_theme(style="whitegrid")

# # قائمة الأعمدة المراد فحص قيمها الشاذة وتوزيعها
# columns_to_analyze = ['course_credits', 'semester_reg_credits','attempt_number', 'total_fail_credits','total_pass_credits','start_total_in_credits','semester_reg_courses','reg_total_semesters']

# for col in columns_to_analyze:
#     # 1. حساب الأرقام الدقيقة للقيم الشاذة باستخدام IQR
#     q1 = df_feature2[col].quantile(0.25)
#     q3 = df_feature2[col].quantile(0.75)
#     iqr = q3 - q1
#     lower_bound = q1 - 1.5 * iqr
#     upper_bound = q3 + 1.5 * iqr
    
#     # تحديد الحسابات الشاذة
#     outliers = df_feature2[(df_feature2[col] < lower_bound) | (df_feature2[col] > upper_bound)]
#     num_outliers = len(outliers)
#     pct_outliers = (num_outliers / len(df_feature2)) * 100
    
#     # 2. إنشاء الشكل ذو اللوحتين (Two Plots)
#     fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [2, 1]})
#     fig.suptitle(f"Analysis for Column: '{col}'", fontsize=16, fontweight='bold', y=1.02)
    
#     # اللوحة الأولى (يسار): مخطط التوزيع ورؤية التكتلات (KDE + Histogram)
#     sns.histplot(data=df_feature2, x=col, kde=True, ax=axes[0], color='darkblue', bins=50)
#     axes[0].set_title("Data Distribution & Grouping Clusters", fontsize=12)
#     axes[0].set_xlabel(col)
#     axes[0].set_ylabel("Count")
    
#     # اللوحة الثانية (يمين): مخطط الصندوق لإظهار النقاط الشاذة بوضوح (Boxplot)
#     sns.boxplot(data=df_feature2, y=col, ax=axes[1], color='darkred', flierprops={"marker": "x", "markerfacecolor": "red"})
#     axes[1].set_title("Boxplot (Outliers Visualized as 'x')", fontsize=12)
#     axes[1].set_ylabel(col)
    
#     # إضافة نص توضيحي على الرسم يبين عدد القيم الشاذة وحدود المنطق
#     info_text = (
#         f"Total Rows: {len(df_feature2):,}\n"
#         f"Lower Bound: {lower_bound:.2f}\n"
#         f"Upper Bound: {upper_bound:.2f}\n"
#         f"Outliers Count: {num_outliers:,}\n"
#         f"Outliers Pct: {pct_outliers:.2f}%"
#     )
#     axes[1].text(1.05, 0.5, info_text, transform=axes[1].transAxes, fontsize=11,
#                  bbox=dict(boxstyle="round,pad=0.5", facecolor="wheat", alpha=0.5))
    
#     plt.tight_layout()
#     plt.show()
    
#     # طباعة التقرير في واجهة الكود (Console)
#     print(f"=== Report for {col} ===")
#     print(f"Number of Outliers: {num_outliers:,} out of {len(df_feature2):,} ({pct_outliers:.2f}%)")
#     print(f"Healthy Range: [{lower_bound} to {upper_bound}]\n" + "="*40 + "\n")

In [50]:
df_feature2.corr(numeric_only=True)

,part_id,final_mark,course_credits,attempt_number,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
part_id,1.000000,0.182058,0.050009,-0.062347,0.201163,0.228159,0.148070,0.218567,0.229018,0.039781,-0.015627,0.144861,0.207819,-0.036819,0.180426,0.877948,-0.032927,-0.022969,0.218125,0.142804
final_mark,0.182058,1.000000,-0.047475,-0.252085,0.489831,0.754374,0.255794,0.129062,0.130922,0.086672,0.092826,0.463955,0.105581,-0.268738,-0.067147,0.200996,-0.066462,-0.039997,0.224481,0.025144
course_credits,0.050009,-0.047475,1.000000,0.022058,-0.024328,-0.014823,0.023478,0.020228,0.127884,0.241673,-0.233830,0.155128,0.130404,0.068345,0.081778,0.014241,0.328341,0.277465,0.093898,0.002826
attempt_number,-0.062347,-0.252085,0.022058,1.000000,-0.380812,-0.301296,-0.088271,-0.008047,-0.006146,-0.182608,-0.197523,-0.298740,0.019021,0.401095,0.201551,-0.143797,0.020786,0.008347,-0.117911,-0.021372
prev_gpa_points,0.201163,0.489831,-0.024328,-0.380812,1.000000,0.631793,0.479264,0.174809,0.171694,0.159453,0.175547,0.429661,0.130077,-0.400572,-0.135486,0.253302,-0.000335,0.062917,0.254809,0.019056
gpa_points,0.228159,0.754374,-0.014823,-0.301296,0.631793,1.000000,0.328260,0.175475,0.174583,0.102086,0.127616,0.629152,0.140426,-0.352266,-0.084122,0.251564,-0.006194,0.042349,0.271972,0.028133
start_agpa_points,0.148070,0.255794,0.023478,-0.088271,0.479264,0.328260,1.000000,0.424553,0.419096,0.050896,0.039907,0.219437,0.399502,-0.043948,0.272914,0.007275,0.183885,0.190097,0.132518,-0.087420
start_total_in_courses,0.218567,0.129062,0.020228,-0.008047,0.174809,0.175475,0.424553,1.000000,0.969096,0.021718,0.094913,0.132953,0.946562,0.246830,0.826771,-0.185662,0.268612,0.300274,0.074040,-0.131098
start_total_in_credits,0.229018,0.130922,0.127884,-0.006146,0.171694,0.174583,0.419096,0.969096,1.000000,0.086004,0.023566,0.167977,0.978963,0.265662,0.830695,-0.174442,0.332853,0.318108,0.107913,-0.129277
semester_reg_credits,0.039781,0.086672,0.241673,-0.182608,0.159453,0.102086,0.050896,0.021718,0.086004,1.000000,0.683906,0.709358,0.079898,-0.031916,0.000941,0.021544,0.139332,0.161616,0.186480,0.016022


In [51]:
df_feature2['prev_gpa_points'].value_counts(dropna=False)

prev_gpa_points
<NA>    92121
0.0     37382
2.5     13645
2.25    13356
2.75    12072
        ...  
3.91       24
3.93       13
0.09       11
3.87        9
3.97        5
Name: count, Length: 391, dtype: Int64

In [52]:
d = pd.read_parquet(RAW_DIR / "v_add_student_degree_status.parquet")

In [53]:
d[d['student_id'].eq(130.111)].head(10)

,student_status_id,student_id,part_id,degree_id,start_part_id,finish_part_id,grade_version_id,permanent_status_id,permanent_status_sl,study_mode,prev_gpa_points,prev_gpa_percent,gpa_percent,gpa_points,start_agpa_percent,start_agpa_points,start_total_in_courses,start_total_in_credits,end_total_in_courses,end_total_in_credits,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,semester_in_courses,semester_in_credits,total_semesters,total_reg_courses,total_reg_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,degree_name_sl,degree_credits_count,start_level_id,start_level_name_pl
23,150532.111,130.111,20111.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,NaN,NaN,0.0,0.00,20.0,0.00,0.0,0.0,0.0,0.0,20.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
24,150533.111,130.111,20112.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,20.0,0.00,0.0,0.0,0.0,0.0,20.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,7.0,17.0,6.0,14.0,1.0,3.0,2.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
25,150534.111,130.111,20121.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,20.0,0.00,0.0,0.0,0.0,0.0,20.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,15.0,35.0,11.0,26.0,4.0,9.0,3.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
26,203129.111,130.111,20122.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,20.0,0.00,0.0,0.0,0.0,0.0,20.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,16.0,38.0,11.0,26.0,5.0,12.0,3.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
27,150535.111,130.111,20131.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,20.0,0.00,0.0,0.0,0.0,0.0,20.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,16.0,38.0,11.0,26.0,5.0,12.0,4.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
28,150536.111,130.111,20132.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,20.0,0.00,0.0,0.0,0.0,0.0,20.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,23.0,56.0,18.0,44.0,5.0,12.0,5.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
29,150537.111,130.111,20141.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,65.6,2.28,20.0,0.00,0.0,0.0,23.0,55.0,65.6,2.28,7.0,17.0,6.0,14.0,1.0,3.0,23.0,55.0,7.0,30.0,73.0,25.0,61.0,5.0,12.0,6.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,252.111,Second Year
30,150538.111,130.111,20142.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,2.28,65.6,0.0,0.00,65.6,2.28,23.0,55.0,23.0,55.0,42.2,1.11,7.0,18.0,0.0,0.0,7.0,18.0,0.0,0.0,8.0,37.0,90.0,31.0,75.0,6.0,15.0,7.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,253.111,Third year
31,146766.111,130.111,20151.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.0,0.00,42.2,1.11,23.0,55.0,23.0,55.0,42.2,1.11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,44.0,108.0,31.0,75.0,13.0,33.0,7.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,253.111,Third year
32,214209.111,130.111,20152.0,10.111,20111.0,20152.0,2.111,17.0,ترقين قيد,C,0.00,0.0,0.0,0.00,42.2,1.11,23.0,55.0,23.0,55.0,42.2,1.11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,44.0,108.0,31.0,75.0,13.0,33.0,7.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,253.111,Third year


In [54]:
d[d['start_agpa_points'].eq(0)]

,student_status_id,student_id,part_id,degree_id,start_part_id,finish_part_id,grade_version_id,permanent_status_id,permanent_status_sl,study_mode,prev_gpa_points,prev_gpa_percent,gpa_percent,gpa_points,start_agpa_percent,start_agpa_points,start_total_in_courses,start_total_in_credits,end_total_in_courses,end_total_in_credits,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,semester_in_courses,semester_in_credits,total_semesters,total_reg_courses,total_reg_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,degree_name_sl,degree_credits_count,start_level_id,start_level_name_pl
3,114381.111,128.111,20111.0,19.111,20111.0,20151.0,2.111,2.0,مقبول,C,NaN,NaN,51.40,1.57,20.0,0.0,0.0,0.0,5.0,13.0,51.40,1.57,7.0,17.0,5.0,13.0,2.0,4.0,5.0,13.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة الأعمال,148.0,236.111,First Year
13,113735.111,129.111,20111.0,19.111,20111.0,20143.0,2.111,2.0,مقبول,C,NaN,NaN,56.88,1.51,0.0,0.0,0.0,0.0,5.0,12.0,56.88,1.51,7.0,17.0,5.0,12.0,2.0,5.0,5.0,12.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,GRADUATED,القرار الوزاري الموحد,إدارة الأعمال,148.0,236.111,First Year
23,150532.111,130.111,20111.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,NaN,NaN,0.00,0.00,20.0,0.0,0.0,0.0,0.0,0.0,20.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
24,150533.111,130.111,20112.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.00,0.00,20.0,0.0,0.0,0.0,0.0,0.0,20.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,7.0,17.0,6.0,14.0,1.0,3.0,2.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
25,150534.111,130.111,20121.0,10.111,20111.0,20152.0,2.111,2.0,مقبول,C,0.00,0.0,0.00,0.00,20.0,0.0,0.0,0.0,0.0,0.0,20.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,15.0,35.0,11.0,26.0,4.0,9.0,3.0,CLOSE_FILE,القرار الوزاري الموحد,إدارة المؤسسات المالية و المصرفية,134.0,251.111,First Year
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189153,413198.111,33701.111,20252.0,50.111,20252.0,NaN,3.111,2.0,مقبول,C,NaN,NaN,76.00,2.80,0.0,0.0,0.0,0.0,6.0,16.0,76.00,2.80,6.0,16.0,6.0,16.0,0.0,0.0,6.0,16.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,القرار رقم 230,إجازة في هندسة البترول2023,180.0,1987.111,First Year
189154,413222.111,33702.111,20252.0,50.111,20252.0,NaN,3.111,2.0,مقبول,C,NaN,NaN,43.40,1.17,0.0,0.0,0.0,0.0,3.0,7.0,43.40,1.17,5.0,13.0,3.0,7.0,2.0,6.0,3.0,7.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,القرار رقم 230,إجازة في هندسة البترول2023,180.0,1987.111,First Year
189155,417770.111,33702.111,20253.0,50.111,20252.0,NaN,3.111,2.0,مقبول,C,1.17,43.4,0.00,0.00,0.0,0.0,3.0,7.0,3.0,7.0,43.40,1.17,2.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,5.0,13.0,3.0,7.0,2.0,6.0,2.0,NaN,القرار رقم 230,إجازة في هندسة البترول2023,180.0,1987.111,First Year
189156,413225.111,33703.111,20252.0,50.111,20252.0,NaN,3.111,2.0,مقبول,C,NaN,NaN,54.20,1.71,0.0,0.0,0.0,0.0,4.0,10.0,54.20,1.71,5.0,12.0,4.0,10.0,1.0,2.0,4.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,القرار رقم 230,إجازة في هندسة البترول2023,180.0,1987.111,First Year


In [55]:
a=d[(d['part_id'].eq(20164)) & (d['semester_in_credits'] > 24)]

In [56]:
len(a)

16

In [57]:
a.head()

,student_status_id,student_id,part_id,degree_id,start_part_id,finish_part_id,grade_version_id,permanent_status_id,permanent_status_sl,study_mode,prev_gpa_points,prev_gpa_percent,gpa_percent,gpa_points,start_agpa_percent,start_agpa_points,start_total_in_courses,start_total_in_credits,end_total_in_courses,end_total_in_credits,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,semester_in_courses,semester_in_credits,total_semesters,total_reg_courses,total_reg_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,degree_name_sl,degree_credits_count,start_level_id,start_level_name_pl
11213,236500.111,3680.111,20164.0,2.111,20121.0,20191.0,3.111,2.0,مقبول,C,1.87,57.4,58.0,1.90,61.4,2.07,49.0,173.0,54.0,198.0,62.0,2.10,5.0,25.0,5.0,25.0,0.0,0.0,5.0,25.0,16.0,57.0,203.0,49.0,173.0,8.0,30.0,15.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,283.111,Sixth Year
11707,236996.111,3700.111,20164.0,2.111,20121.0,20181.0,3.111,2.0,مقبول,C,2.50,70.0,59.4,1.97,65.8,2.29,60.0,222.0,60.0,222.0,67.4,2.37,8.0,36.0,7.0,32.0,1.0,4.0,7.0,32.0,15.0,65.0,241.0,65.0,241.0,0.0,0.0,15.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,283.111,Sixth Year
12033,236645.111,3715.111,20164.0,2.111,20121.0,20181.0,3.111,2.0,مقبول,C,1.75,55.0,42.8,1.14,60.0,2.00,59.0,213.5,60.0,219.5,61.0,2.05,9.0,42.0,6.0,28.0,3.0,14.0,6.0,28.0,18.0,72.0,267.5,65.0,239.5,8.0,30.0,18.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,283.111,Sixth Year
12316,236543.111,3730.111,20164.0,2.111,20121.0,20181.0,3.111,2.0,مقبول,C,2.50,70.0,62.2,2.11,68.2,2.41,61.0,224.0,61.0,224.0,69.2,2.46,6.0,28.0,6.0,28.0,0.0,0.0,6.0,28.0,18.0,77.0,284.0,71.0,261.0,6.0,23.0,18.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,283.111,Sixth Year
13104,236922.111,3777.111,20164.0,2.111,20131.0,20201.0,3.111,2.0,مقبول,C,0.55,31.0,50.4,1.52,55.0,1.75,39.0,130.5,45.0,156.0,59.4,1.97,7.0,30.5,6.0,25.5,1.0,5.0,6.0,25.5,13.0,56.0,198.0,42.0,142.5,14.0,55.5,13.0,GRADUATED,القرار رقم 230,دكتور في الطب,251.0,282.111,Fifth year


In [58]:
mask = d["semester_reg_credits"].gt(25)

print("Rows with semester_reg_credits > 24:", mask.sum())
print("Unique students:", d.loc[mask, "student_id"].nunique())

Rows with semester_reg_credits > 24: 813
Unique students: 675


In [59]:
d.loc[mask, ["student_id", "part_id", "degree_id", "semester_reg_credits"]].drop_duplicates()

,student_id,part_id,degree_id,semester_reg_credits
2057,840.111,20112.0,11.111,34.0
2938,1674.111,20123.0,13.111,53.0
2941,1674.111,20141.0,13.111,29.0
3141,1843.111,20164.0,13.111,37.0
3207,1846.111,20154.0,13.111,26.0
...,...,...,...,...
124962,16475.111,20232.0,2.111,27.0
126967,16863.111,20252.0,2.111,27.0
128496,17148.111,20252.0,2.111,27.0
132828,17673.111,20251.0,2.111,27.0


In [60]:
d1 = pd.read_parquet(RAW_DIR / "v_crg_student_course_raw.parquet")

In [61]:
d1[(d1['student_id'].eq(840.111))&(d1['part_id'].eq(20112.0))]

,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,in_credits,in_gpa,in_agpa,study_mode,degree_id,student_name_sl,degree_name_sl,faculty_id,course_credits,active
6473,473549.111,840.111,178.111,20112.0,93.111,66.0,1.0,P,سياسة سعرية,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,3.0,A
6474,473550.111,840.111,177.111,20112.0,92.111,68.0,1.7,P,ادارة مخازن ومشتريات,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,2.0,A
6475,473551.111,840.111,172.111,20112.0,91.111,70.0,2.0,P,قانون تجاري,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,2.0,A
6476,473552.111,840.111,176.111,20112.0,89.111,77.0,2.7,P,ادارة مؤسسات تجارية خارجية,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,2.0,A
6477,473553.111,840.111,174.111,20112.0,85.111,100.0,4.0,P,محاسبة شركات 2,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,3.0,A
6770,480718.111,840.111,201.111,20112.0,684.111,0.0,0.0,F,اعلان وبحوث تسويق,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,3.0,A
6771,480719.111,840.111,204.111,20112.0,684.111,0.0,0.0,F,اتصال اداري,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,2.0,A
6772,480720.111,840.111,205.111,20112.0,684.111,0.0,0.0,F,بحوث العمليات,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,4.0,A
6773,480721.111,840.111,203.111,20112.0,684.111,0.0,0.0,F,ادارة الشركات متعددة الجنسية,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,2.0,A
6774,480722.111,840.111,202.111,20112.0,684.111,0.0,0.0,F,احصاء متقدم,R,Y,Y,Y,C,11.111,فراس فارس محمود,المحاسبة و التدقيق,7.111,3.0,A


In [62]:
# Guard (governance contract 13): diploma columns must survive selection.
_required_diploma_cols = ["diploma_gpa", "diploma_type_id"]
_missing_diploma = [c for c in _required_diploma_cols if c not in df_feature2.columns]
assert not _missing_diploma, (
    f"Diploma columns dropped during selection: {_missing_diploma}. "
    "They must be present in the selected output - check the drop lists above."
)

HISTORY_HELPER_COLS = ["gpa_points", "semester_pass_credits"]
missing_history_helpers = [col for col in HISTORY_HELPER_COLS if col not in df_feature2.columns]
if missing_history_helpers:
    raise KeyError(f"selected_model_population is missing history helper columns before save: {missing_history_helpers}")

print("History helper columns before selected_model_population save:")
print(pd.Series({col: col in df_feature2.columns for col in HISTORY_HELPER_COLS}, name="exists").to_string())
print("\nHistory helper non-null counts before selected_model_population save:")
helper_non_null_counts = df_feature2[HISTORY_HELPER_COLS].notna().sum()
print(helper_non_null_counts.to_string())
all_null_history_helpers = helper_non_null_counts[helper_non_null_counts.eq(0)].index.tolist()
if all_null_history_helpers:
    raise ValueError(f"selected_model_population helper columns are present but all null: {all_null_history_helpers}")
print("\nHistory helper dtypes before selected_model_population save:")
print(df_feature2[HISTORY_HELPER_COLS].dtypes.to_string())

def _derive_validation_university_id(frame):
    derived = pd.Series(pd.NA, index=frame.index, dtype="string")
    for candidate_col in ["student_id", "degree_id", "course_id", "faculty_id", "grade_id", "student_course_id"]:
        if candidate_col not in frame.columns:
            continue
        values = frame[candidate_col].astype("string").str.strip()
        suffix = values.str.extract(r"\.([^.]+)$", expand=False)
        derived = derived.fillna(suffix)
    return derived.fillna("__MISSING__")

stability_validation_frame = df_feature2.copy()
if "university_id" in stability_validation_frame.columns:
    semester_stability_keys = ["university_id", "student_id", "degree_id", "part_id"]
else:
    stability_validation_frame["_validation_university_id"] = _derive_validation_university_id(stability_validation_frame)
    semester_stability_keys = ["_validation_university_id", "student_id", "degree_id", "part_id"]
    print("\nSemester stability uses derived university_id from dotted ID suffixes for validation only.")

helper_conflict_counts = {}
helper_conflict_examples = []
for helper_col in HISTORY_HELPER_COLS:
    unique_counts = stability_validation_frame.groupby(semester_stability_keys, dropna=False)[helper_col].nunique(dropna=False)
    conflicts = unique_counts[unique_counts.gt(1)]
    helper_conflict_counts[helper_col] = int(len(conflicts))
    if not conflicts.empty:
        helper_conflict_examples.append(
            conflicts.reset_index(name="unique_value_count").assign(column=helper_col).head(10)
        )

print("\nHistory helper semester-level stability conflicts before selected_model_population save:")
print(pd.Series(helper_conflict_counts, name="conflicting_semester_groups").to_string())
if helper_conflict_examples:
    display(pd.concat(helper_conflict_examples, ignore_index=True).head(25))

save_parquet(df_feature2, FEATURES_DIR / "selected_model_population.parquet")

History helper columns before selected_model_population save:
gpa_points               True
semester_pass_credits    True

History helper non-null counts before selected_model_population save:
gpa_points               784245
semester_pass_credits    784245

History helper dtypes before selected_model_population save:
gpa_points               Float64
semester_pass_credits    Float64

Semester stability uses derived university_id from dotted ID suffixes for validation only.

History helper semester-level stability conflicts before selected_model_population save:
gpa_points               0
semester_pass_credits    0


WindowsPath('D:/AI/Real projects/Academic_Advisor/data/features/selected_model_population.parquet')

## Pandas copy/view audit — 01_select_model_population.ipynb

Two patterns fixed in this notebook:

| Cell | Before | After | Reason |
|---|---|---|---|
| `df1.dropna(...)` | `df1.dropna(subset=[...], inplace=True)` | `df1 = df1.dropna(subset=[...])` | Avoid inplace on df from `.drop()` — use assignment form |
| `df_feature2.drop(...)` | `df_feature2.drop(columns=[...], inplace=True)` | `df_feature2 = df_feature2.drop(columns=[...])` | Same — avoid inplace, use assignment form |

All other DataFrames in this notebook (`df1`, `df_feature`, `df_feature2`) are results of `.drop()` or `.read_parquet()` and are already independent copies. No `.copy()` additions were needed elsewhere.